In [18]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import os 
from padelpy import padeldescriptor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from rdkit.Chem import rdFingerprintGenerator  
import pandas as pd 
from padelpy import from_smiles
import warnings
warnings.filterwarnings('ignore')

In [20]:
df = pd.read_csv('cleaned_data.csv')
df.head()

,SMILES,VERIFY,ESOL Class,MW_pub,BBB,hERG,Ames,H-HT
0,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,False,Moderately soluble,468.4,0.982377,0.138762,0.684111,0.349169
1,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,False,Moderately soluble,476.3,0.999029,0.262471,0.351038,0.586856
2,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,True,Moderately soluble,472.4,0.989686,0.112066,0.887385,0.556827
3,C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C(=C2)F)F)[C@...,True,Moderately soluble,486.4,0.001523,0.666432,0.854345,0.977018
4,C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C=C2)F)[C@@]3...,False,Soluble,430.4,0.004441,0.332171,0.871076,0.912706


In [21]:
df.isnull().sum()  # Check for any remaining NaN values in the DataFrame

SMILES        0
VERIFY        0
ESOL Class    0
MW_pub        0
BBB           0
hERG          0
Ames          0
H-HT          0
dtype: int64

In [22]:
from rdkit import Chem

# Function to convert SMILES → canonical SMILES
def to_canonical(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None   # invalid SMILES
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None

# Apply to your dataframe column
df['canonical_smiles'] = df['SMILES'].apply(to_canonical)

# Check results
print(df[['SMILES', 'canonical_smiles']].head())


                                              SMILES  \
0  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   
1  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   
2  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   
3  C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C(=C2)F)F)[C@...   
4  C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C=C2)F)[C@@]3...   

                                    canonical_smiles  
0  COC1CCC2(CC1)Cc1ccc(B3OC(C)(C)C(C)(C)O3)cc1C21...  
1  CN1C(=O)NC2(C1=O)c1cc(B3OC(C)(C)C(C)(C)O3)ccc1...  
2  COC1CCC2(CC1)Cc1ccc(B3OC(C)(C)C(C)(C)O3)cc1C21...  
3  C#CCOc1ccc(C(=O)Nc2cc(F)c(F)c([C@]3(CF)C[C@@H]...  
4  C#CCOc1ccc(C(=O)Nc2ccc(F)c([C@@]3(C(F)F)N=C(N)...  


In [ ]:
# df['canonical_smiles']

print(df['SMILES'].duplicated().sum())
print(df['canonical_smiles'].shape)
df['canonical_smiles'].duplicated().sum()
# 1592
# (2856,)
# 1592

596
(1860,)


596

In [25]:
# Check results
df[['SMILES', 'canonical_smiles']].head()

,SMILES,canonical_smiles
0,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,COC1CCC2(CC1)Cc1ccc(B3OC(C)(C)C(C)(C)O3)cc1C21...
1,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,CN1C(=O)NC2(C1=O)c1cc(B3OC(C)(C)C(C)(C)O3)ccc1...
2,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,COC1CCC2(CC1)Cc1ccc(B3OC(C)(C)C(C)(C)O3)cc1C21...
3,C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C(=C2)F)F)[C@...,C#CCOc1ccc(C(=O)Nc2cc(F)c(F)c([C@]3(CF)C[C@@H]...
4,C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C=C2)F)[C@@]3...,C#CCOc1ccc(C(=O)Nc2ccc(F)c([C@@]3(C(F)F)N=C(N)...


In [28]:
# import pandas as pd
# from padelpy import from_smiles

# df = df.dropna(subset=['canonical_smiles'])

# # Calculate descriptors for all canonical SMILES
# descriptors = from_smiles(df['canonical_smiles'].tolist(), 
#                           fingerprints=True,   # include PubChem fingerprints
#                           threads=2)           # use 2 CPU threads

# # Convert to DataFrame
# desc_df = pd.DataFrame(descriptors)

# # Merge back with original dataset (keeping cid as identifier)
# df_desc = pd.concat([df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)

# print(df_desc.shape)
# df_desc.head()


In [30]:
import os
import time
import pandas as pd
from padelpy import padeldescriptor

# # 1. Clean data and drop missing values
# df = df.dropna(subset=['canonical_smiles'])

# 2. Set up explicit, static filenames in your current directory
current_dir = os.getcwd()
temp_smi_path = os.path.join(current_dir, "windows_padel_input.smi")
temp_csv_path = os.path.join(current_dir, "windows_padel_output.csv")

# Write the canonical SMILES directly to the static file
df['canonical_smiles'].to_csv(temp_smi_path, index=False, header=False)

print("Running PaDEL-Descriptor safely...")
try:
    # We call padeldescriptor directly, bypassing the broken from_smiles internal code
    padeldescriptor(
        mol_dir=temp_smi_path,
        d_file=temp_csv_path,
        fingerprints=True,      # Include PubChem fingerprints
        detectaromaticity=True,
        standardizenitro=True,
        standardizetautomers=True,
        threads=2,              # Use 2 CPU threads
        removesalt=True,
        log=False
    )
    
    # 3. Read the calculated descriptors back
    desc_df = pd.read_csv(temp_csv_path)
    
    # Drop the structural tracking column generated by PaDEL
    if 'Name' in desc_df.columns:
        desc_df = desc_df.drop(columns=['Name'])
        
    # 4. Merge back cleanly with your original dataset
    df_desc = pd.concat([df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)
    print(f"Calculations complete! Final data shape: {df_desc.shape}")
    
    # Display the final combined dataframe
    print(df_desc.head())

except Exception as e:
    print(f"Execution failed: {e}")

finally:
    # 5. Safe Cleanup Loop with a cooldown period for Windows
    time.sleep(1.5) 
    if os.path.exists(temp_smi_path):
        try: os.remove(temp_smi_path)
        except: pass
    if os.path.exists(temp_csv_path):
        try: os.remove(temp_csv_path)
        except: pass

Running PaDEL-Descriptor safely...
Calculations complete! Final data shape: (1860, 890)
                                              SMILES  VERIFY  \
0  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   False   
1  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   False   
2  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...    True   
3  C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C(=C2)F)F)[C@...    True   
4  C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C=C2)F)[C@@]3...   False   

           ESOL Class  MW_pub       BBB      hERG      Ames      H-HT  \
0  Moderately soluble   468.4  0.982377  0.138762  0.684111  0.349169   
1  Moderately soluble   476.3  0.999029  0.262471  0.351038  0.586856   
2  Moderately soluble   472.4  0.989686  0.112066  0.887385  0.556827   
3  Moderately soluble   486.4  0.001523  0.666432  0.854345  0.977018   
4             Soluble   430.4  0.004441  0.332171  0.871076  0.912706   

                                    canonical_smiles  PubchemFP0  ...  \
0  COC1CCC2(CC1

In [32]:
df_desc.to_csv('clean_padel_descriptors.csv', index=False)

In [34]:
cpd = pd.read_csv('clean_padel_descriptors.csv')
cpd.head()

,SMILES,VERIFY,ESOL Class,MW_pub,BBB,hERG,Ames,H-HT,canonical_smiles,PubchemFP0,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,False,Moderately soluble,468.4,0.982377,0.138762,0.684111,0.349169,COC1CCC2(CC1)Cc1ccc(B3OC(C)(C)C(C)(C)O3)cc1C21...,0,...,0,0,0,0,0,0,0,0,0,0
1,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,False,Moderately soluble,476.3,0.999029,0.262471,0.351038,0.586856,CN1C(=O)NC2(C1=O)c1cc(B3OC(C)(C)C(C)(C)O3)ccc1...,0,...,0,0,0,0,0,0,0,0,0,0
2,B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...,True,Moderately soluble,472.4,0.989686,0.112066,0.887385,0.556827,COC1CCC2(CC1)Cc1ccc(B3OC(C)(C)C(C)(C)O3)cc1C21...,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C(=C2)F)F)[C@...,True,Moderately soluble,486.4,0.001523,0.666432,0.854345,0.977018,C#CCOc1ccc(C(=O)Nc2cc(F)c(F)c([C@]3(CF)C[C@@H]...,1,...,0,0,0,0,0,0,0,0,0,0
4,C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C=C2)F)[C@@]3...,False,Soluble,430.4,0.004441,0.332171,0.871076,0.912706,C#CCOc1ccc(C(=O)Nc2ccc(F)c([C@@]3(C(F)F)N=C(N)...,1,...,0,0,0,0,0,0,0,0,0,0
